In [ ]:
import os, sys
from pathlib import Path
REPO_ROOT = Path("D:/repos/pla")
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from log_utils.logging import read_logs
import numpy as np

In [ ]:
# This code gets the min, max, avg performace stats 
#  - for each simulation timestep,
#  - in all clients,
#  - from the beginning of the simulation unitl the timestep where drift starts

def compute_pre_drift_stats_all_clients(client_log, drift_start):
    """
    Pre-drift stats: for each round t in [0, drift_start),
    compute min/max/avg across ALL clients.
    """
    pre_stats = {}
    for t in range(drift_start):
        losses = [loss for loss, _ in client_log[t]]
        accs   = [acc  for _, acc in client_log[t]]

        pre_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return pre_stats

In [ ]:
# This is a helper function

def extend_clusters_for_final_segment(clusters_per_segment):
    """
    compute_cluster_segmented_stats creates an extra segment [last_drift, T),
    so we reuse the last clustering pattern for that final segment.
    """
    return clusters_per_segment + [clusters_per_segment[-1]]

In [ ]:
def reorder_clusters_by_pattern(
    drift_clustered_client_indices,
    drift_pattern,
    keep_unmentioned="append"  # "append" | "drop"
):
    """
    Reorders clusters at each step according to drift_pattern.

    Supports:
      - subset orders like [1]
      - repeated indices like [1,1]
      - normal permutations like [2,1]

    keep_unmentioned:
      - "append": append clusters not referenced in 'order' at the end (in original order)
      - "drop": only keep clusters referenced in 'order'
    """
    if len(drift_clustered_client_indices) != len(drift_pattern):
        raise ValueError("Length mismatch: clustered indices and pattern must have same #steps.")

    reordered = []

    for step_clusters, order in zip(drift_clustered_client_indices, drift_pattern):
        k = len(step_clusters)

        # Validate each requested cluster index
        for idx in order:
            if not (1 <= idx <= k):
                raise ValueError(f"Invalid cluster index {idx} for step with {k} clusters (valid: 1..{k}).")

        # Reorder (1-based -> 0-based). Allows duplicates.
        reordered_step = [step_clusters[i - 1] for i in order]

        if keep_unmentioned == "append":
            mentioned = set(order)
            # append unmentioned clusters in original order
            for i in range(1, k + 1):
                if i not in mentioned:
                    reordered_step.append(step_clusters[i - 1])
        elif keep_unmentioned == "drop":
            pass
        else:
            raise ValueError("keep_unmentioned must be 'append' or 'drop'.")

        reordered.append(reordered_step)

    return reordered


In [ ]:
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    """
    client_log[t][client] = (loss, acc)
    Computes per-round min/max/avg over the given 'clients' for t in [start_t, end_t).
    """
    per_round_cluster_stats  = {}
    for t in range(start_t, end_t):
        losses = [client_log[t][c][0] for c in clients]
        accs   = [client_log[t][c][1] for c in clients]

        per_round_cluster_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return per_round_cluster_stats 

In [ ]:
def compute_cluster_segmented_stats(client_log, drift_steps, clusters_per_segment):
    """
    Compute 
        - cluster-wise 
        - per-round statistics, for,
        - each drift segment, 
        - INCLUDING the final segment from the last drift step to the end of the simulation.

    Inputs
    ------
    client_log:
        Nested list where client_log[t][client_id] = (loss, accuracy)
    drift_steps:
        List of drift timestamps, e.g., [20, 33, 35, 45]
        Segments are built as: [20,33), [33,35), [35,45), [45,T)
    clusters_per_segment:
        List aligned with segments. Each element contains clusters for that segment:
          clusters_per_segment[segment_idx][cluster_idx] = [client_ids...]

        IMPORTANT: Must include cluster definitions for the final [last_drift, T) segment as well.

    Output
    ------
    cluster_stats:
        cluster_stats[cluster_idx][(start, end)][t] = {
            "loss_min", "loss_max", "loss_avg",
            "acc_min",  "acc_max",  "acc_avg"
        }
    """
    T = len(client_log)

    # Ensure drift steps are in increasing order so segment construction is correct.
    drift_steps_sorted = sorted(drift_steps)

    if not drift_steps_sorted:
        raise ValueError("drift_steps is empty; cannot build drift segments without at least one drift step.")

    # Build segments between drift points PLUS the final segment to T:
    # e.g., [20,33), [33,35), [35,45), [45,T)
    segment_boundaries = drift_steps_sorted + [T]
    drift_segments = list(zip(segment_boundaries[:-1], segment_boundaries[1:]))

    # clusters_per_segment must provide cluster definitions for EACH segment (including the final one).
    if len(drift_segments) != len(clusters_per_segment):
        raise ValueError(
            f"Mismatch: {len(drift_segments)} drift segments but {len(clusters_per_segment)} cluster definitions."
        )

    # We assume the number of clusters is consistent across segments (e.g., always 3 after appending [8,9]).
    n_clusters = len(clusters_per_segment[0])
    for segment_idx, segment_clusters in enumerate(clusters_per_segment):
        if len(segment_clusters) != n_clusters:
            raise ValueError(
                "All segments must have the same number of clusters "
                "(e.g., after appending [8,9]). "
                f"Segment {segment_idx} has {len(segment_clusters)} clusters, expected {n_clusters}."
            )

    # Prepare output container:
    cluster_stats = {cluster_idx: {} for cluster_idx in range(n_clusters)}

    # For each segment, compute per-round stats for each cluster's client IDs.
    for (segment_start, segment_end), segment_clusters in zip(drift_segments, clusters_per_segment):
        for cluster_idx, client_ids in enumerate(segment_clusters):
            cluster_stats[cluster_idx][(segment_start, segment_end)] = per_round_stats_for_clients_in_range(
                client_log=client_log,
                clients=client_ids,
                start_t=segment_start,
                end_t=segment_end
            )

    return cluster_stats

In [ ]:
# For easy plotting

def cluster_stats_to_plot_series(cluster_stats):
    """
    Converts:
      cluster_stats[k][(start,end)][t] = stats
    into:
      plot_data[k][(start,end)] = {"round": [...], "acc_avg":[...], "loss_avg":[...], ...}
    """
    plot_data = {}

    for k, seg_dict in cluster_stats.items():
        plot_data[k] = {}
        for (start, end), per_round in seg_dict.items():
            rounds = sorted(per_round.keys())
            plot_data[k][(start, end)] = {
                "round": rounds,
                "loss_min": [per_round[r]["loss_min"] for r in rounds],
                "loss_avg": [per_round[r]["loss_avg"] for r in rounds],
                "loss_max": [per_round[r]["loss_max"] for r in rounds],
                "acc_min":  [per_round[r]["acc_min"]  for r in rounds],
                "acc_avg":  [per_round[r]["acc_avg"]  for r in rounds],
                "acc_max":  [per_round[r]["acc_max"]  for r in rounds],
            }
    return plot_data

In [ ]:
def build_plot_inputs_for_log(client_log, drift_steps, clusters_per_segment_without_final):
    """
    Returns:
      pre_stats: dict[t] -> stats
      post_cluster_plot_stats: output of cluster_stats_to_plot_series(cluster_stats)
    """
    drift_steps_sorted = sorted(drift_steps)
    drift_start = drift_steps_sorted[0]

    pre_stats = compute_pre_drift_stats_all_clients(client_log, drift_start)

    clusters_per_segment = extend_clusters_for_final_segment(clusters_per_segment_without_final)

    cluster_stats = compute_cluster_segmented_stats(
        client_log=client_log,
        drift_steps=drift_steps_sorted,
        clusters_per_segment=clusters_per_segment
    )

    post_cluster_plot_stats = cluster_stats_to_plot_series(cluster_stats)
    return pre_stats, post_cluster_plot_stats

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def stats_dict_to_arrays(stats_by_round, metric_prefix):
    rounds = np.array(sorted(stats_by_round.keys()), dtype=int)
    y_min = np.array([stats_by_round[t][f"{metric_prefix}_min"] for t in rounds], dtype=float)
    y_avg = np.array([stats_by_round[t][f"{metric_prefix}_avg"] for t in rounds], dtype=float)
    y_max = np.array([stats_by_round[t][f"{metric_prefix}_max"] for t in rounds], dtype=float)
    return rounds, y_min, y_avg, y_max

In [ ]:
def plot_pre_post_drift_continuous_on_axis(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    metric_prefix,                  # "acc" or "loss"
    base_color,                     # one color per method
    pre_label,
    pre_linestyle="-",              # solid for pre-drift
    pre_linewidth=1.8,
    cluster_linestyles=None,        # dict: {cluster_idx: linestyle}
    cluster_labels=None,            # dict: {cluster_idx: label}
    cluster_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,               # list of drift boundaries (vertical lines)
    collect_legend_handles=True,    # whether to collect legend handles for this method
    legend_mode="method",           # "method" | "method+clusters" | "none"
):
    """
    Adds pre-drift + post-drift cluster curves to an existing axis.
    Returns a list of legend handles according to legend_mode.
    """
    if cluster_linestyles is None:
        cluster_linestyles = {}
    if cluster_labels is None:
        cluster_labels = {}

    legend_handles = []

    # -------------------------
    # 1) Pre-drift (all clients)
    # -------------------------
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    pre_line, = ax.plot(
        x_pre, y_pre_avg,
        zorder=4 if base_color == "green" else 3, alpha=0.6, color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        label=pre_label
    )

    if collect_legend_handles and legend_mode in ("method", "method+clusters"):
        legend_handles.append(pre_line)

    if show_band:
        ax.fill_between(
            x_pre, y_pre_min, y_pre_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    # Identify the last pre-drift round/value for stitching
    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # -------------------------
    # 2) Find first post-drift round
    # -------------------------
    first_post_round = None
    for _, segments_dict in post_cluster_plot_stats.items():
        for (_, _), seg in segments_dict.items():
            if len(seg["round"]) > 0:
                candidate = int(seg["round"][0])
                first_post_round = candidate if first_post_round is None else min(first_post_round, candidate)

    # -------------------------
    # 3) Stitch pre -> post
    # -------------------------
    if stitch and first_post_round is not None and (pre_last_round + 1 == first_post_round):
        for _, segments_dict in post_cluster_plot_stats.items():
            cluster_first_avg = None
            for (_, _), seg in segments_dict.items():
                if len(seg["round"]) == 0:
                    continue
                if int(seg["round"][0]) == first_post_round:
                    cluster_first_avg = float(seg[f"{metric_prefix}_avg"][0])
                    break

            if cluster_first_avg is None:
                continue

            # Stitching line uses the SAME style as pre-drift
            ax.plot(
                [pre_last_round, first_post_round],
                [pre_last_avg, cluster_first_avg],
                zorder=4 if base_color == "green" else 3, alpha=0.6, color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth
            )

    # -------------------------
    # 4) Drift boundary markers
    # -------------------------
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(
                x=d,
                color="black",
                linestyle=":",
                linewidth=0.8,
                alpha=0.6,
                zorder=0
            )

    # -------------------------
    # 5) Post-drift (cluster-wise), continuous across segments
    # -------------------------
    for cluster_idx, segments_dict in post_cluster_plot_stats.items():
        linestyle = cluster_linestyles.get(cluster_idx, "--")
        label = cluster_labels.get(cluster_idx, f"{pre_label} | Cluster {cluster_idx}")

        x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []

        for (start, end) in sorted(segments_dict.keys()):
            seg = segments_dict[(start, end)]
            x_seg = np.array(seg["round"], dtype=int)

            y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
            y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
            y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

            x_all.append(x_seg)
            y_min_all.append(y_min_seg)
            y_avg_all.append(y_avg_seg)
            y_max_all.append(y_max_seg)

        if not x_all:
            continue

        x = np.concatenate(x_all)
        y_min = np.concatenate(y_min_all)
        y_avg = np.concatenate(y_avg_all)
        y_max = np.concatenate(y_max_all)

        cluster_line, = ax.plot(
            x, y_avg,
            zorder=4 if base_color == "green" else 3, alpha=0.6, color=base_color,
            linestyle=linestyle,
            linewidth=cluster_linewidth,
            label=label
        )

        # Only include cluster lines in legend when explicitly requested
        if collect_legend_handles and legend_mode == "method+clusters":
            legend_handles.append(cluster_line)

        if show_band:
            ax.fill_between(
                x, y_min, y_max,
                color=base_color,
                alpha=band_alpha,
                linewidth=0
            )

    return legend_handles

In [ ]:
def plot_multiple_methods_pre_post_drift(
    *,
    method_configs,
    drift_steps,
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
):
    """
    Plots multiple methods (logs) on the same Accuracy and Loss figures and saves both.

    method_configs: list of dicts, each:
      {
        "name": "L1",
        "pre_stats": <dict>,
        "post_cluster_plot_stats": <dict>,
        "color": "red",
        "cluster_linestyles": {0:"--",1:"-.",2:":"},
        "cluster_labels": {0:"...",1:"...",2:"..."}  (optional)
        "show_in_legend": True/False                (optional, default True)
        "legend_mode": "method" | "method+clusters" | "none"  (optional, default "method")
      }
    """
    drift_steps_sorted = sorted(drift_steps)

    fig_acc, ax_acc = plt.subplots()
    ax_acc.set_axisbelow(True)
    ax_acc.grid(True, linestyle=":", color="gray", alpha=0.3)
    fig_loss, ax_loss = plt.subplots()
    ax_loss.set_axisbelow(True)
    ax_loss.grid(True, linestyle=":", color="gray", alpha=0.3)

    acc_handles_all = []
    loss_handles_all = []

    for cfg_idx, cfg in enumerate(method_configs):
        pre_stats = cfg["pre_stats"]
        post_stats = cfg["post_cluster_plot_stats"]

        base_color = cfg.get("color", "black")
        name = cfg.get("name", "method")

        cluster_linestyles = cfg.get("cluster_linestyles", {})
        cluster_labels = cfg.get("cluster_labels", {})

        collect_legend = cfg.get("show_in_legend", True)
        legend_mode = cfg.get("legend_mode", "method")

        # Draw drift lines only once (on the first method) to avoid over-plotting
        drift_lines_for_this_call = drift_steps_sorted if cfg_idx == 0 else None

        acc_handles = plot_pre_post_drift_continuous_on_axis(
            ax_acc,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="acc",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        loss_handles = plot_pre_post_drift_continuous_on_axis(
            ax_loss,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="loss",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            pre_linewidth=1.8,
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        # Legend collection (no duplicates)
        if collect_legend and acc_handles and loss_handles:
            if legend_mode == "method+clusters":
                acc_handles_all.extend(acc_handles)
                loss_handles_all.extend(loss_handles)
            elif legend_mode == "method":
                acc_handles_all.append(acc_handles[0])
                loss_handles_all.append(loss_handles[0])
            # legend_mode == "none" -> add nothing

    annotate_scenario_intervals(ax_acc, drift_steps_sorted, drift_pattern)
    annotate_scenario_intervals(ax_loss, drift_steps_sorted, drift_pattern)

    # Save both figures once (using your function)
    plt.figure(fig_acc.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label="Accuracy",
        _title="",
        _file_path=f"{out_prefix}_accuracy",
        # _legend_handles=acc_handles_all  # Enable to restore the in-plot legend.
        _legend_handles=None
    )

    plt.figure(fig_loss.number)
    configure_and_save_plot(
        _plt=plt,
        plot_formats=("png", "pdf", "pgf"),
        _x_label="Round",
        _y_label="Loss",
        _title="",
        _file_path=f"{out_prefix}_loss",
        # _legend_handles=loss_handles_all  # Enable to restore the in-plot legend.
        _legend_handles=None
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os
import warnings

def configure_and_save_plot(_plt, _x_label, _y_label, _title, _file_path, _legend_handles=None, plot_formats=("png", "pdf", "pgf")):
    _plt.xlabel(_x_label)
    _plt.ylabel(_y_label)
    if _legend_handles is not None:
        _plt.legend(handles=_legend_handles, loc="best", fontsize=4)
    os.makedirs(os.path.dirname(_file_path), exist_ok=True)
    if "png" in plot_formats: _plt.savefig(_file_path + ".png", dpi=300)
    if "pdf" in plot_formats: _plt.savefig(_file_path + ".pdf", format="pdf")
    if "pgf" in plot_formats:
        try: _plt.savefig(_file_path + ".pgf", format="pgf")
        except Exception as exc: warnings.warn(f"PGF export failed: {exc}", RuntimeWarning)
    _plt.close()

def annotate_drift_intervals(ax, boundaries, labels=("2", "3"), arrow_y=0.16, label_y=0.12):
    transform = ax.get_xaxis_transform()
    for left, right, label in zip(boundaries[:-1], boundaries[1:], labels):
        ax.annotate("", xy=(right, arrow_y), xytext=(left, arrow_y), xycoords=transform, textcoords=transform, arrowprops=dict(arrowstyle="<->", color="black", linewidth=1.2, mutation_scale=12, shrinkA=0, shrinkB=0), zorder=6)
        ax.text((left + right) / 2, label_y, rf"$T_{{{label}}}$", transform=transform, ha="center", va="top", fontsize=24, color="black", zorder=6)

def annotate_scenario_intervals(ax, boundaries, drift_pattern):
    if drift_pattern == [[1, 2], [1, 2]]:
        annotate_drift_intervals(ax, boundaries, labels=("2", "3"))
# Crop outer whitespace in every export, retaining 0.1-inch padding.
plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1})

# Double the previous 10-point axis/tick fonts.
plt.rcParams.update({"font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})


# CIFAR-10 learning-rate ablation plots

This notebook applies the plotting and styling functions from `3_1_pala_drifted_clients_plots-CIFAR-10.ipynb` to each `lN` run found in the ablation directory. Every run is overlaid on the same accuracy and loss figures and labeled L1, L2, etc.

In [ ]:
from pathlib import Path
from log_utils.logging import read_logs
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Keep the source notebook font, grid, opacity, and export settings.
plt.rcParams.update({"savefig.bbox": "tight", "savefig.pad_inches": 0.1})
plt.rcParams.update({"font.size": 20, "axes.labelsize": 20,
                     "xtick.labelsize": 20, "ytick.labelsize": 20})

## Discover ablation logs and load drift metadata

In [ ]:
LOG_ROOT = Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR_rot_l_ablation/rot_l_ablation")

# Prefer the newest nested l_ablation run group when present.
RUN_ROOT = LOG_ROOT / "l_ablation" if (LOG_ROOT / "l_ablation").is_dir() else LOG_ROOT
# Include numbered lN folders only; ignore any other folders starting with l.
log_dirs = sorted(
    (path for path in RUN_ROOT.iterdir() if path.is_dir() and path.name.lower().startswith("l") and path.name[1:].isdigit() and (path / "client_log.pkl").is_file()),
    key=lambda path: int(path.name[1:]),
)
if not log_dirs:
    raise FileNotFoundError(f"No lN log folders found under {LOG_ROOT}")

# Read shared drift specifications wherever they are stored under this root.
metadata_candidates = list(RUN_ROOT.rglob("drift_specs_log.pkl"))
if not metadata_candidates:
    raise FileNotFoundError(f"No drift_specs_log.pkl found under {RUN_ROOT}")
metadata_path = metadata_candidates[0]
drift_specs = read_logs(str(metadata_path))
drift_steps = drift_specs["drift_step_rounds"]
drift_clustered_client_indices = drift_specs["drift_clustered_client_indices"]

# Scenario B pattern used by the source CIFAR-10 notebook.
drift_pattern = [[1, 2], [1, 2]]
reordered_clusters = reorder_clusters_by_pattern(drift_clustered_client_indices, drift_pattern)
extra_cluster = [8, 9]
reordered_clusters_complete = [groups + [extra_cluster] for groups in reordered_clusters]
clusters_no_final = reordered_clusters_complete

print("Discovered runs:", [path.name for path in log_dirs])
print("Drift rounds:", drift_steps)

## Load each run and prepare common plot inputs

In [ ]:
SMOOTH_WINDOW = 6

def smooth_client_log(client_log, window=SMOOTH_WINDOW):
    """Apply a centered moving average to each client's loss and accuracy series."""
    if window < 1:
        raise ValueError("Smoothing window must be at least one round.")
    if window == 1:
        return client_log

    result = []
    left = (window - 1) // 2
    right = window // 2
    for round_idx in range(len(client_log)):
        lo = max(0, round_idx - left)
        hi = min(len(client_log), round_idx + right + 1)
        round_values = []
        for client_idx in range(len(client_log[0])):
            losses = [client_log[t][client_idx][0] for t in range(lo, hi)]
            accuracies = [client_log[t][client_idx][1] for t in range(lo, hi)]
            round_values.append((float(np.mean(losses)), float(np.mean(accuracies))))
        result.append(round_values)
    return result

In [ ]:
cluster_linestyles = {0: "--", 1: "-.", 2: ":"}
MODEL_LAYERS = ("conv1", "conv2", "fc1", "fc2")
method_configs = []

for index, log_dir in enumerate(log_dirs, start=1):
    run_log_path = log_dir / "client_log.pkl"
    if not run_log_path.exists():
        raise FileNotFoundError(f"Expected client log at {run_log_path}")

    raw_run_log = read_logs(str(run_log_path))
    run_log = smooth_client_log(raw_run_log, SMOOTH_WINDOW)
    pre_stats, post_stats = build_plot_inputs_for_log(
        run_log, drift_steps, clusters_no_final
    )
    method_configs.append({
        "name": f"L{len(MODEL_LAYERS) - int(log_dir.name[1:])}: {', '.join(MODEL_LAYERS[:int(log_dir.name[1:])]) or 'none'}",
        "pre_stats": pre_stats,
        "post_cluster_plot_stats": post_stats,
        "color": None,
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method",
    })

## Plot all learning-rate runs together

In [ ]:
RAW_OUT_PREFIX = "D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/4_1_l_ablation_CIFAR-10"
SMOOTH_OUT_PREFIX = "D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/smooth/4_1_l_ablation_CIFAR-10"
color_cycle = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]  # L1-L4 palette used by the CIFAR-10 L-ablation figures.
for index, cfg in enumerate(method_configs):
    cfg["color"] = color_cycle[index % len(color_cycle)]

def save_ablation_legend(configs, file_prefix):
    handles = [Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6, label=cfg["name"]) for cfg in configs]
    fig = plt.figure(figsize=(max(2.4, len(handles) * 1.05), 0.5))
    fig.legend(handles=handles, labels=[cfg["name"] for cfg in configs], loc="center", ncol=len(handles), frameon=False, fontsize=14, handlelength=2.0, columnspacing=0.8, borderaxespad=0.0, labelspacing=0.1, handletextpad=0.25)
    plt.axis("off")
    os.makedirs(os.path.dirname(file_prefix), exist_ok=True)
    fig.savefig(file_prefix + ".png", dpi=300, bbox_inches="tight", pad_inches=0.01)
    fig.savefig(file_prefix + ".pdf", bbox_inches="tight", pad_inches=0.01)
    try:
        fig.savefig(file_prefix + ".pgf", format="pgf", bbox_inches="tight", pad_inches=0.01)
    except Exception as exc:
        warnings.warn(f"PGF legend export failed: {exc}", RuntimeWarning)
    plt.close(fig)

raw_method_configs = []
for index, log_dir in enumerate(log_dirs, start=1):
    raw_log = read_logs(str(log_dir / "client_log.pkl"))
    pre_stats, post_stats = build_plot_inputs_for_log(raw_log, drift_steps, clusters_no_final)
    raw_method_configs.append({
        "name": f"L{len(MODEL_LAYERS) - int(log_dir.name[1:])}: {', '.join(MODEL_LAYERS[:int(log_dir.name[1:])]) or 'none'}",
        "pre_stats": pre_stats,
        "post_cluster_plot_stats": post_stats,
        "color": color_cycle[(index - 1) % len(color_cycle)],
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method",
    })

plot_multiple_methods_pre_post_drift(
    method_configs=raw_method_configs,
    drift_steps=drift_steps,
    out_prefix=RAW_OUT_PREFIX,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
)

# Matching standalone legends for the raw and smoothed curve sets.
save_ablation_legend(raw_method_configs, RAW_OUT_PREFIX + "_legend")
save_ablation_legend(raw_method_configs, RAW_OUT_PREFIX + "_nL_legend")
raw_short_legend = [dict(cfg, name=cfg["name"].split(":", 1)[0]) for cfg in raw_method_configs]
save_ablation_legend(raw_short_legend, RAW_OUT_PREFIX + "_nL_only_legend")

plot_multiple_methods_pre_post_drift(
    method_configs=method_configs,
    drift_steps=drift_steps,
    out_prefix=SMOOTH_OUT_PREFIX,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
)
save_ablation_legend(method_configs, SMOOTH_OUT_PREFIX + "_legend")
save_ablation_legend(method_configs, SMOOTH_OUT_PREFIX + "_nL_legend")
smooth_short_legend = [dict(cfg, name=cfg["name"].split(":", 1)[0]) for cfg in method_configs]
save_ablation_legend(smooth_short_legend, SMOOTH_OUT_PREFIX + "_nL_only_legend")

# Repeat both plot sets without the min/max variance shading; suffix: _nv.
plot_multiple_methods_pre_post_drift(method_configs=raw_method_configs, drift_steps=drift_steps, out_prefix=RAW_OUT_PREFIX + "_nv", band_alpha=0.25, show_band=False, stitch=True)
plot_multiple_methods_pre_post_drift(method_configs=method_configs, drift_steps=drift_steps, out_prefix=SMOOTH_OUT_PREFIX + "_nv", band_alpha=0.25, show_band=False, stitch=True)


### Smoothed no-variance accuracy plot without the y-axis label

In [ ]:
# Additional smoothed CIFAR-10 accuracy plot: no variance band and no y-axis label.
# Uses the six-round-smoothed method_configs prepared above.
fig, ax = plt.subplots()
_drift_steps_sorted = sorted(drift_steps)
for _config_index, _config in enumerate(method_configs):
    plot_pre_post_drift_continuous_on_axis(
        ax,
        pre_stats=_config["pre_stats"],
        post_cluster_plot_stats=_config["post_cluster_plot_stats"],
        metric_prefix="acc",
        base_color=_config["color"],
        pre_label=_config["name"],
        pre_linestyle="-",
        cluster_linestyles=_config.get("cluster_linestyles", {}),
        cluster_labels=_config.get("cluster_labels", {}),
        band_alpha=0.25,
        show_band=False,
        stitch=True,
        drift_steps=_drift_steps_sorted if _config_index == 0 else None,
        collect_legend_handles=False,
        legend_mode="none",
    )

annotate_scenario_intervals(ax, _drift_steps_sorted, drift_pattern)
ax.set_xlabel("Round")
# Intentionally omit ax.set_ylabel("Accuracy").
ax.grid(True, linestyle=":", color="gray", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
_NO_YLABEL_PREFIX = Path(
    "D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/smooth"
) / "4_1_l_ablation_CIFAR-10_nv_accuracy_wo_ax_label"
os.makedirs(str(_NO_YLABEL_PREFIX.parent), exist_ok=True)
for _format in ("png", "pdf", "pgf"):
    try:
        fig.savefig(str(_NO_YLABEL_PREFIX) + "." + _format,
                    format=_format if _format != "png" else None,
                    dpi=300 if _format == "png" else None,
                    bbox_inches="tight", pad_inches=0.1)
    except Exception as _exc:
        if _format == "pgf":
            warnings.warn("PGF export failed for {}: {}".format(_NO_YLABEL_PREFIX, _exc), RuntimeWarning)
        else:
            raise
plt.close(fig)
print("Saved", str(_NO_YLABEL_PREFIX))


In [ ]:
# Regenerate the compact L-only legend in the requested output folder.
from matplotlib.lines import Line2D
from pathlib import Path
import os
import warnings

legend_configs = [dict(cfg, name=cfg["name"].split(":", 1)[0]) for cfg in method_configs]
legend_colors = plt.rcParams["axes.prop_cycle"].by_key().get("color", ["red", "blue", "green", "darkviolet"])
for index, cfg in enumerate(legend_configs):
    cfg["color"] = legend_colors[index % len(legend_colors)]
legend_handles = [
    Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6, label=cfg["name"])
    for cfg in legend_configs
]
legend_fig = plt.figure(figsize=(max(2.4, len(legend_handles) * 1.05), 0.5))
legend_fig.legend(
    handles=legend_handles, labels=[cfg["name"] for cfg in legend_configs],
    loc="center", ncol=len(legend_handles), frameon=False, fontsize=14,
    handlelength=2.0, columnspacing=0.8, borderaxespad=0.0,
    labelspacing=0.1, handletextpad=0.25,
)
legend_fig.patch.set_alpha(0)
legend_prefix = Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_l_ablation/4_1_l_ablation_CIFAR-10_nL_only_legend")
legend_prefix.parent.mkdir(parents=True, exist_ok=True)
legend_fig.savefig(str(legend_prefix) + ".png", dpi=300, bbox_inches="tight", pad_inches=0.01, transparent=True)
legend_fig.savefig(str(legend_prefix) + ".pdf", bbox_inches="tight", pad_inches=0.01, transparent=True)
try:
    legend_fig.savefig(str(legend_prefix) + ".pgf", format="pgf", bbox_inches="tight", pad_inches=0.01, transparent=True)
except Exception as exc:
    warnings.warn(f"PGF legend export failed: {exc}", RuntimeWarning)
plt.close(legend_fig)
print("Saved compact L-only legend:", legend_prefix)


In [ ]:
# Regenerate only the smoothed rotation L-ablation outputs with the canonical L-ablation palette.
from pathlib import Path
from matplotlib.lines import Line2D
import os
import warnings

ABLATION_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
if len(method_configs) > len(ABLATION_COLORS):
    raise ValueError("Add a color to ABLATION_COLORS for every L-run")
for _idx, _cfg in enumerate(method_configs):
    _cfg["color"] = ABLATION_COLORS[_idx]

ROT_L_SMOOTH_PREFIX = Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/smooth/4_1_l_ablation_CIFAR-10")
plot_multiple_methods_pre_post_drift(
    method_configs=method_configs, drift_steps=drift_steps,
    out_prefix=str(ROT_L_SMOOTH_PREFIX), band_alpha=0.25,
    show_band=True, stitch=True,
)
plot_multiple_methods_pre_post_drift(
    method_configs=method_configs, drift_steps=drift_steps,
    out_prefix=str(ROT_L_SMOOTH_PREFIX) + "_nv", band_alpha=0.25,
    show_band=False, stitch=True,
)

def _save_rot_l_legend(configs, output_prefix):
    _handles = [Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6, label=cfg["name"])
                for cfg in configs]
    _fig = plt.figure(figsize=(max(2.4, len(_handles) * 1.05), 0.5))
    _fig.legend(_handles, [cfg["name"] for cfg in configs], loc="center",
                ncol=len(_handles), frameon=False, fontsize=14, handlelength=2.0,
                columnspacing=0.8, borderaxespad=0.0, labelspacing=0.1, handletextpad=0.25)
    os.makedirs(os.path.dirname(output_prefix), exist_ok=True)
    _fig.savefig(output_prefix + ".png", dpi=300, bbox_inches="tight", pad_inches=0.01)
    _fig.savefig(output_prefix + ".pdf", bbox_inches="tight", pad_inches=0.01)
    try:
        _fig.savefig(output_prefix + ".pgf", format="pgf", bbox_inches="tight", pad_inches=0.01)
    except Exception as exc:
        warnings.warn("PGF legend export failed: {}".format(exc), RuntimeWarning)
    plt.close(_fig)

_save_rot_l_legend(method_configs, str(ROT_L_SMOOTH_PREFIX) + "_legend")
_save_rot_l_legend(method_configs, str(ROT_L_SMOOTH_PREFIX) + "_nL_legend")
_rot_l_short_legend = [dict(cfg, name=cfg["name"].split(":", 1)[0]) for cfg in method_configs]
_save_rot_l_legend(_rot_l_short_legend, str(ROT_L_SMOOTH_PREFIX) + "_nL_only_legend")

# Preserve the existing no-variance, no-y-label accuracy export with the same colors.
_fig, _ax = plt.subplots()
for _i, _cfg in enumerate(method_configs):
    plot_pre_post_drift_continuous_on_axis(
        _ax, pre_stats=_cfg["pre_stats"],
        post_cluster_plot_stats=_cfg["post_cluster_plot_stats"],
        metric_prefix="acc", base_color=_cfg["color"], pre_label=_cfg["name"],
        pre_linestyle="-", cluster_linestyles=_cfg.get("cluster_linestyles", {}),
        cluster_labels=_cfg.get("cluster_labels", {}), band_alpha=0.25,
        show_band=False, stitch=True,
        drift_steps=sorted(drift_steps) if _i == 0 else None,
        collect_legend_handles=False, legend_mode="none",
    )
annotate_scenario_intervals(_ax, sorted(drift_steps), drift_pattern)
_ax.set_xlabel("Round")
_ax.set_ylabel("")
_ax.grid(True, linestyle=":", color="gray", alpha=0.3)
_ax.set_axisbelow(True)
_configure_fig = _ax.figure
_configure_fig.tight_layout()
_configure_fig.savefig(str(ROT_L_SMOOTH_PREFIX) + "_nv_accuracy_wo_ax_label.png", dpi=300, bbox_inches="tight", pad_inches=0.1)
_configure_fig.savefig(str(ROT_L_SMOOTH_PREFIX) + "_nv_accuracy_wo_ax_label.pdf", bbox_inches="tight", pad_inches=0.1)
try:
    _configure_fig.savefig(str(ROT_L_SMOOTH_PREFIX) + "_nv_accuracy_wo_ax_label.pgf", format="pgf", bbox_inches="tight", pad_inches=0.1)
except Exception as exc:
    warnings.warn("PGF export failed: {}".format(exc), RuntimeWarning)
plt.close(_configure_fig)
print("Regenerated smoothed rotation L-ablation figures with canonical colors in:", ROT_L_SMOOTH_PREFIX.parent)


In [ ]:
# Generate six-round-smoothed Drift 2 plots for rotation and label-swap L-ablation runs.
from pathlib import Path
import matplotlib.pyplot as plt

DRIFT2_COLOR_BY_RUN = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c", 4: "#d62728"}
DRIFT2_SCENARIO_PATTERN = [[1, 2], [1, 2]]
DRIFT2_RUNS = [
    {
        "kind": "rotation",
        "log_root": Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR_rot_l_ablation/rot_l_ablation"),
        "output_root": Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/smooth/drift2"),
    },
    {
        "kind": "label_swap",
        "log_root": Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR-10_l_ablation"),
        "output_root": Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_l_ablation/smooth/drift_2"),
    },
]

def _drift2_find_run_root(log_root):
    nested = log_root / "l_ablation"
    for candidate in (nested, log_root):
        if candidate.is_dir() and any(
            child.is_dir() and child.name.lower().startswith("l")
            and child.name[1:].isdigit() and (child / "client_log.pkl").is_file()
            for child in candidate.iterdir()
        ):
            return candidate
    raise FileNotFoundError("No numbered lN client logs found under {}".format(log_root))

def _drift2_plot_one_metric(configs, drift_boundaries, output_path, metric_prefix, show_ylabel=False):
    fig, ax = plt.subplots()
    ax.set_axisbelow(True)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    for cfg_index, cfg in enumerate(configs):
        # Keep the pre-drift series and only the Drift 2 post-drift cluster.
        plot_pre_post_drift_continuous_on_axis(
            ax, pre_stats=cfg["pre_stats"],
            post_cluster_plot_stats={1: cfg["post_cluster_plot_stats"][1]},
            metric_prefix=metric_prefix, base_color=cfg["color"],
            pre_label=cfg["name"], pre_linestyle="-",
            pre_linewidth=2.7, cluster_linewidth=2.7,
            cluster_linestyles={1: "-."}, cluster_labels={1: cfg["name"]},
            band_alpha=0.25, show_band=False, stitch=True,
            drift_steps=drift_boundaries if cfg_index == 0 else None,
            collect_legend_handles=False, legend_mode="none",
        )
    annotate_scenario_intervals(ax, sorted(drift_boundaries), DRIFT2_SCENARIO_PATTERN)
    configure_and_save_plot(
        _plt=plt, _x_label="Round",
        _y_label=("Accuracy" if metric_prefix == "acc" else "Loss") if show_ylabel else "",
        _title="", _file_path=str(output_path), _legend_handles=None,
        plot_formats=("png", "pdf", "pgf"),
    )

for _run in DRIFT2_RUNS:
    _run_root = _drift2_find_run_root(_run["log_root"])
    _log_dirs = sorted(
        (p for p in _run_root.iterdir() if p.is_dir() and p.name.lower().startswith("l")
         and p.name[1:].isdigit() and (p / "client_log.pkl").is_file()),
        key=lambda p: int(p.name[1:]),
    )
    if set(int(p.name[1:]) for p in _log_dirs) != set(DRIFT2_COLOR_BY_RUN):
        raise ValueError("Expected l1-l4 folders in {}".format(_run_root))
    _metadata_path = next(iter(sorted(_run_root.rglob("drift_specs_log.pkl"))), None)
    if _metadata_path is None:
        raise FileNotFoundError("No drift_specs_log.pkl found under {}".format(_run_root))
    _spec = read_logs(str(_metadata_path))
    _drift_steps = list(_spec["drift_step_rounds"])
    if len(_drift_steps) != 3:
        raise ValueError("Scenario B Drift 2 requires three boundaries; found {}".format(_drift_steps))
    _reordered = reorder_clusters_by_pattern(
        _spec["drift_clustered_client_indices"], DRIFT2_SCENARIO_PATTERN)
    _reference_log = read_logs(str(_log_dirs[0] / "client_log.pkl"))
    _client_ids = set(range(len(_reference_log[0])))
    _drifted_ids = {client for step in _reordered for group in step for client in group}
    _no_drift_clients = sorted(_client_ids - _drifted_ids)
    _clusters_per_segment = [groups + [_no_drift_clients] for groups in _reordered]

    _plot_configs = []
    for _log_dir in _log_dirs:
        _run_id = int(_log_dir.name[1:])
        _raw = read_logs(str(_log_dir / "client_log.pkl"))
        _smoothed = smooth_client_log(_raw, SMOOTH_WINDOW)
        _pre, _post = build_plot_inputs_for_log(
            _smoothed, _drift_steps, _clusters_per_segment)
        _layer_count = _run_id
        _name = "L{}: {}".format(
            len(MODEL_LAYERS) - _layer_count,
            ", ".join(MODEL_LAYERS[:_layer_count]) or "none")
        _plot_configs.append({
            "name": _name, "pre_stats": _pre,
            "post_cluster_plot_stats": _post,
            "color": DRIFT2_COLOR_BY_RUN[_run_id],
        })

    _run["output_root"].mkdir(parents=True, exist_ok=True)
    for _metric in ("acc", "loss"):
        if _metric == "acc":
            _suffix = "rot_accuracy" if _run["kind"] == "rotation" else "wap_accuracy"
        else:
            _suffix = "loss"
        _out = _run["output_root"] / "4_1_l_ablation_CIFAR-10_drift_2_{}_wo_ax_label".format(_suffix)
        _drift2_plot_one_metric(_plot_configs, _drift_steps, _out, _metric, show_ylabel=False)
    print("Generated smoothed Drift 2 {} plots from {} in {}".format(
        _run["kind"], _run_root, _run["output_root"]))


In [ ]:
# Generate six-round-smoothed Drift 1 average-only plots for rotation and label-swap L-ablation runs.
# Thick lines, average curves only; no min/max deviation bands.
# No min/max deviation bands are drawn; average curves and Scenario B T2/T3 remain.
DRIFT1_RUNS = [
    {
        "kind": "rotation",
        "log_root": Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR_rot_l_ablation/rot_l_ablation"),
        "output_root": Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_rot_l_ablation/smooth/drift1"),
    },
    {
        "kind": "label_swap",
        "log_root": Path("D:/publications_related/2026/ICDCS/logs/swap/seed42/B/CIFAR-10_l_ablation"),
        "output_root": Path("D:/publications_related/2026/ICDCS/figures/new/seed42/B/CIFAR-10_l_ablation/smooth/drift1"),
    },
]

def _drift1_plot_one_metric(configs, drift_boundaries, output_path, metric_prefix, show_ylabel=True):
    fig, ax = plt.subplots()
    ax.set_axisbelow(True)
    ax.grid(True, linestyle=":", color="gray", alpha=0.3)
    for cfg_index, cfg in enumerate(configs):
        # Keep pre-drift performance and only the Drift 1 cluster afterward.
        plot_pre_post_drift_continuous_on_axis(
            ax, pre_stats=cfg["pre_stats"],
            post_cluster_plot_stats={0: cfg["post_cluster_plot_stats"][0]},
            metric_prefix=metric_prefix, base_color=cfg["color"],
            pre_label=cfg["name"], pre_linestyle="-",
            pre_linewidth=2.7, cluster_linewidth=2.7,
            cluster_linestyles={0: "--"}, cluster_labels={0: cfg["name"]},
            band_alpha=0.25, show_band=False, stitch=True,
            drift_steps=drift_boundaries if cfg_index == 0 else None,
            collect_legend_handles=False, legend_mode="none",
        )
    annotate_scenario_intervals(ax, sorted(drift_boundaries), DRIFT2_SCENARIO_PATTERN)
    configure_and_save_plot(
        _plt=plt, _x_label="Round",
        _y_label=("Accuracy" if metric_prefix == "acc" else "Loss") if show_ylabel else "",
        _title="", _file_path=str(output_path), _legend_handles=None,
        plot_formats=("png", "pdf", "pgf"),
    )

for _run in DRIFT1_RUNS:
    _run_root = _drift2_find_run_root(_run["log_root"])
    _log_dirs = sorted(
        (p for p in _run_root.iterdir() if p.is_dir() and p.name.lower().startswith("l")
         and p.name[1:].isdigit() and (p / "client_log.pkl").is_file()),
        key=lambda p: int(p.name[1:]),
    )
    if set(int(p.name[1:]) for p in _log_dirs) != set(DRIFT2_COLOR_BY_RUN):
        raise ValueError("Expected l1-l4 folders in {}".format(_run_root))
    _metadata_path = next(iter(sorted(_run_root.rglob("drift_specs_log.pkl"))), None)
    if _metadata_path is None:
        raise FileNotFoundError("No drift_specs_log.pkl found under {}".format(_run_root))
    _spec = read_logs(str(_metadata_path))
    _drift_steps = list(_spec["drift_step_rounds"])
    if len(_drift_steps) != 3:
        raise ValueError("Scenario B Drift 1 requires three boundaries; found {}".format(_drift_steps))
    _reordered = reorder_clusters_by_pattern(
        _spec["drift_clustered_client_indices"], DRIFT2_SCENARIO_PATTERN)
    _reference_log = read_logs(str(_log_dirs[0] / "client_log.pkl"))
    _client_ids = set(range(len(_reference_log[0])))
    _drifted_ids = {client for step in _reordered for group in step for client in group}
    _no_drift_clients = sorted(_client_ids - _drifted_ids)
    _clusters_per_segment = [groups + [_no_drift_clients] for groups in _reordered]

    _plot_configs = []
    for _log_dir in _log_dirs:
        _run_id = int(_log_dir.name[1:])
        _raw = read_logs(str(_log_dir / "client_log.pkl"))
        _smoothed = smooth_client_log(_raw, SMOOTH_WINDOW)
        _pre, _post = build_plot_inputs_for_log(
            _smoothed, _drift_steps, _clusters_per_segment)
        _name = "L{}: {}".format(
            len(MODEL_LAYERS) - _run_id,
            ", ".join(MODEL_LAYERS[:_run_id]) or "none")
        _plot_configs.append({
            "name": _name, "pre_stats": _pre,
            "post_cluster_plot_stats": _post,
            "color": DRIFT2_COLOR_BY_RUN[_run_id],
        })

    _run["output_root"].mkdir(parents=True, exist_ok=True)
    for _metric in ("acc", "loss"):
        _metric_name = "accuracy" if _metric == "acc" else "loss"
        if _metric == "acc":
            _suffix = "rot_accuracy" if _run["kind"] == "rotation" else "wap_accuracy"
            _out = _run["output_root"] / "4_1_l_ablation_CIFAR-10_drift_1_{}".format(_suffix)
        else:
            _out = _run["output_root"] / "4_1_l_ablation_CIFAR-10_drift_1_loss"
        _drift1_plot_one_metric(_plot_configs, _drift_steps, _out, _metric)
        if _run["kind"] == "rotation" and _metric == "acc":
            _without_ylabel = _run["output_root"] / "4_1_l_ablation_CIFAR-10_drift_1_rot_accuracy_wo_ax_label"
            _drift1_plot_one_metric(_plot_configs, _drift_steps, _without_ylabel, _metric, show_ylabel=False)
    print("Generated smoothed average-only Drift 1 {} plots from {} in {}".format(
        _run["kind"], _run_root, _run["output_root"]))




In [ ]:
# Regenerate only the rotation nL legend with a tight crop.
from pathlib import Path
from matplotlib.lines import Line2D
import warnings

_ROT_NL_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
if len(method_configs) > len(_ROT_NL_COLORS):
    raise ValueError("Add a color for every L-ablation run.")
_ROT_NL_CONFIGS = [dict(cfg, color=_ROT_NL_COLORS[index])
                   for index, cfg in enumerate(method_configs)]
_ROT_NL_HANDLES = [
    Line2D([0], [0], color=cfg["color"], linewidth=3.6, alpha=0.6,
           label=cfg["name"])
    for cfg in _ROT_NL_CONFIGS
]
_ROT_NL_FIG = plt.figure(figsize=(max(2.4, len(_ROT_NL_HANDLES) * 1.05), 0.5))
_ROT_NL_LEGEND = _ROT_NL_FIG.legend(
    handles=_ROT_NL_HANDLES,
    labels=[cfg["name"] for cfg in _ROT_NL_CONFIGS],
    loc="center", ncol=len(_ROT_NL_HANDLES), frameon=False, fontsize=14,
    handlelength=2.0, columnspacing=0.8, borderaxespad=0.0,
    labelspacing=0.1, handletextpad=0.25,
)
_ROT_NL_FIG.canvas.draw()
_ROT_NL_BBOX = _ROT_NL_LEGEND.get_window_extent(
    renderer=_ROT_NL_FIG.canvas.get_renderer()
).transformed(_ROT_NL_FIG.dpi_scale_trans.inverted())
_ROT_NL_PREFIX = Path(
    "D:/publications_related/2026/ICDCS/figures/new/seed42/B/"
    "CIFAR-10_rot_l_ablation/smooth/4_1_l_ablation_CIFAR-10_nL_legend"
)
_ROT_NL_PREFIX.parent.mkdir(parents=True, exist_ok=True)
_ROT_NL_FIG.savefig(str(_ROT_NL_PREFIX) + ".png", dpi=300,
                    bbox_inches=_ROT_NL_BBOX, pad_inches=0)
_ROT_NL_FIG.savefig(str(_ROT_NL_PREFIX) + ".pdf",
                    bbox_inches=_ROT_NL_BBOX, pad_inches=0)
try:
    _ROT_NL_FIG.savefig(str(_ROT_NL_PREFIX) + ".pgf", format="pgf",
                        bbox_inches=_ROT_NL_BBOX, pad_inches=0)
except Exception as exc:
    warnings.warn("PGF legend export failed: {}".format(exc), RuntimeWarning)
plt.close(_ROT_NL_FIG)
print("Saved tightly cropped rotation nL legend:", _ROT_NL_PREFIX)
